# lemonの結果を作成する



In [ ]:
matcher_names = ["bert_mini", "magellan"]
dataset_root_dir = "../../data/lemon/datasets"
model_root_dir = "../../data/lemon/model"
out_root_dir = "../../data/experiments/10_eval/lemon_result"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]
gpu_id = 5

In [ ]:
TARGET_DATASET_ID = 5
TOP_N = 5
TARGET_MATCHER_ID = 1
START_DATA_IDX = None
END_DATA_IDX = None
GPU_ID = 0

In [ ]:
BATCH_SIZE = 512
gpu_id = GPU_ID

In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_id}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [ ]:
import pathlib
import pickle
from typing import List

import tqdm

from pine.dataset import load_dataset
from pine.matcher.magellan_matcher import _load_magellan_model_predict_func
from pine.matcher.transformer_matcher import load_transformer_matcher_func
from pine.entity import Entity, EntityPair
from pine.explainer import AttributionScore


In [ ]:
import lemon
from typing import Callable, List, Tuple
import pandas as pd


def make_lemon_explanation(
    entity_pair: EntityPair,
    proba_fn: Callable,
    kernel: Callable = None,
    n_sample: int = None,
    random_state: int = 0,
) -> Tuple[
    List[AttributionScore],
    List[AttributionScore],
    float,
    float,
    float,
    float,
    List[str],
    List[str],
]:
    # 入力データのフォーマットを合わせる
    df_l, df_r = entity_pair.to_dataframe()
    df_l = df_l.reset_index(drop=True)
    df_r = df_r.reset_index(drop=True)
    dataframe_id_pair = pd.DataFrame(
        {"a.rid": df_l.index.tolist(), "b.rid": df_r.index.tolist()}
    )
    dataframe_id_pair.index.name = "pid"
    # LEMON(粒度=単語(通常のLEMONではフレーズ))
    exp = lemon.explain(
        df_l,
        df_r,
        dataframe_id_pair,
        proba_fn,
        num_features=1000,
        dual_explanation=True,
        estimate_potential=True,
        granularity="tokens",
        num_samples=n_sample,
        token_representation="record-bow",
        token_patterns="[^ ]+",
        explain_attrs=False,
        attribution_method="lime",
        show_progress=False,
        random_state=random_state,
        return_dict=None,
    )
    # matching score
    match_score_org = proba_fn(df_l, df_r, dataframe_id_pair)
    # 結果のフォーマットを合わせる
    tokens_l = []
    tokens_r = []
    attrs_l = []
    attrs_r = []
    for attribution in exp.attributions:
        for source in ["a", "b"]:
            if all(s != source for s, _, _, _ in attribution.positions):
                continue
            if attribution.name is not None:
                string = attribution.name
            else:
                s, attr, attr_or_val, j = attribution.positions[0]
                val = exp.string_representation[(s, attr, attr_or_val)]
                if j is None:
                    if val is None:
                        string = f"<{attr}>" if attr_or_val == "attr" else f"[{attr}]"
                    else:
                        string = "" if val is None else str(val)
                        if len(string) > 33:
                            string = (
                                f"<{attr}>" if attr_or_val == "attr" else f"[{attr}]"
                            )
                else:
                    string = val[j]
            if source == "a":
                tokens_l.append(string)
                attrs_l.append(AttributionScore(None, attribution.weight))
            if source == "b":
                tokens_r.append(string)
                attrs_r.append(AttributionScore(None, attribution.weight))
    return (
        attrs_l,
        attrs_r,
        match_score_org.iat[0],
        0,
        (exp.metadata["a"]["r2_score"] + exp.metadata["b"]["r2_score"]) / 2,
        exp.prediction_score,
        tokens_l,
        tokens_r,
    )


def test_make_lemon_explanation():
    target_dataset_name = dataset_names[TARGET_DATASET_ID]
    target_matcher_name = "magellan"
    target_idx = 9

    dataset = load_dataset(target_dataset_name, dataset_root_dir)

    if target_matcher_name == "magellan":
        matcher_base_func = _load_magellan_model_predict_func(
            target_dataset_name, model_root_dir
        )
    elif target_matcher_name == "bert_mini":
        matcher_base_func = load_transformer_matcher_func(
            target_dataset_name, model_root_dir
        )

    def proba_func(*args, **kw):
        df_score = matcher_base_func(*args, **kw)
        # スコアを規格化 0.0 - 1.0 を -1.0 - 1.0 にする
        df_score = 2 * df_score - 1.0
        return df_score


    l_id, r_id = dataset.test.record_id_pairs.iloc[target_idx].tolist()
    entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
    entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])
    ret = make_lemon_explanation(
        EntityPair(entity_l, entity_r), proba_func, None, None
    )
    sorted_indexes_l = sorted(range(len(ret[0])), key=lambda k: ret[0][k].score, reverse=True)
    sorted_indexes_r = sorted(range(len(ret[1])), key=lambda k: ret[1][k].score, reverse=True)
    print("attrs_l", [ret[0][i] for i in sorted_indexes_l])
    print("attrs_r", [ret[1][i] for i in sorted_indexes_r])
    print("match_score_org", ret[2], (ret[2] - 0.5) * 2 )
    print("intercept", ret[3])
    print("prediction_score", ret[4])
    print("local_pred", ret[5])
    print("tokens_l", [ret[6][i] for i in sorted_indexes_l])
    print("tokens_r", [ret[7][i] for i in sorted_indexes_r])
    print("label", dataset.test.labels.iloc[target_idx])
    return


test_make_lemon_explanation()

In [ ]:
import pathlib
import pickle
from typing import List

import tqdm


def save_lemon_results(
    dataset,
    matcher_func,
    out_dir,
    save_step,
    top_n,
    start_data_idx=None,
    end_data_idx=None,
):
    for i in tqdm.tqdm(range(0, len(dataset.test.record_id_pairs), save_step)):
        # 開始idxよりも前ならskip
        if (
            start_data_idx is not None
            and i < int(start_data_idx / save_step) * save_step
        ):
            print("skip step {}".format(i))
            continue
        # 終了idxよりも後ろならskip
        if (
            end_data_idx is not None
            and int((end_data_idx - 1) / save_step) * save_step < i
        ):
            print("skip step {}".format(i))
            continue

        lemon_results = {}
        out_file_path = pathlib.Path(out_dir) / f"{i}.pickle"

        if out_file_path.exists():
            # ファイルがあればデータを読み込む
            with out_file_path.open("rb") as f:
                lemon_results = pickle.load(f)

        for data_count, (pid, l_id, r_id) in tqdm.tqdm(
            enumerate(
                dataset.test.record_id_pairs.iloc[i : i + save_step].itertuples()
            ),
            total=save_step,
            leave=False,
        ):
            # 開始idxよりも前ならskip
            if start_data_idx is not None and i + data_count < start_data_idx:
                print("skip {}".format(i + data_count))
                continue
            # 終了idxよりも後ろならskip
            if end_data_idx is not None and end_data_idx - 1 < i + data_count:
                print("skip {}".format(i + data_count))
                continue
            if pid not in lemon_results:
                lemon_results[pid] = {}

            entity_l = Entity.from_dataframe(dataset.test.records.a.loc[[l_id]])
            entity_r = Entity.from_dataframe(dataset.test.records.b.loc[[r_id]])

            # オリジナルがなければ作る
            if (None, None) not in lemon_results[pid]:
                ret = make_lemon_explanation(
                    EntityPair(entity_l, entity_r), matcher_func, None, None
                )
                lemon_results[pid][(None, None)] = ret

        with out_file_path.open("wb") as f:
           pickle.dump(lemon_results, f)
        print("save {}".format(str(out_file_path)))
    return True

In [ ]:
import gc

save_step = 100

target_dataset_name = dataset_names[TARGET_DATASET_ID]
dataset = load_dataset(target_dataset_name, dataset_root_dir)
for target_matcher_name in matcher_names:
    print("=======================")
    print(target_dataset_name, target_matcher_name)
    print("=======================")
    if TARGET_MATCHER_ID is not None and target_matcher_name != matcher_names[TARGET_MATCHER_ID]:
        print("SKIP. Because TARGET_MATCHER_NAME={}".format(matcher_names[TARGET_MATCHER_ID]))
        continue
    if target_matcher_name == "magellan":
        matcher_base_func = _load_magellan_model_predict_func(
            target_dataset_name, model_root_dir
        )
    elif target_matcher_name == "bert_mini":
        matcher_base_func = load_transformer_matcher_func(
            target_dataset_name, model_root_dir, BATCH_SIZE
        )
    
    def proba_func(*args, **kw):
        df_score = matcher_base_func(*args, **kw)
        # スコアを規格化 0.0 - 1.0 を -1.0 - 1.0 にする
        df_score = 2 * df_score - 1.0
        return df_score

    out_dir_path = (
        pathlib.Path(out_root_dir) / target_matcher_name / target_dataset_name
    )
    out_dir_path.mkdir(parents=True, exist_ok=True)
    save_lemon_results(dataset, proba_func, out_dir_path, save_step, TOP_N, START_DATA_IDX, END_DATA_IDX)
    del matcher_base_func
    gc.collect()